# Construcción y Ajuste del Modelo QUBO mediante Regresión Ridge Ponderada

El presente cuaderno desarrolla la transformación del espacio continuo de variables de decisión a un modelo discreto compatible con arquitecturas de computación cuántica. El proceso ingiere el conjunto de datos cinemáticos generados mediante el muestreo de Sobol y aplica una discretización basada en intervalos uniformes. Posteriormente, se implementa una regresión polinomial de grado dos con regularización L2 (Ridge). El resultado finaliza con la exportación de los coeficientes del modelo a un archivo binario para su posterior evaluación.

In [2]:
# --- Paso 0: Instalaciones e Importaciones ---
import sys
import subprocess

print("Instalando bibliotecas requeridas (dimod, sympy, cvxpy, dwave-neal, pandas)...")
try:
    subprocess.run([sys.executable, "-m", "pip", "install", "dimod", "sympy", "cvxpy", "dwave-neal", "pandas", "osqp"],
                   check=True, capture_output=True, text=True)
    print("Instalación completada exitosamente.")
except subprocess.CalledProcessError as e:
    print(f"Error durante la instalación: {e.stderr}")

import pickle
import numpy as np
import pandas as pd
import sympy as sp
import time
import cvxpy as cp
import gc
from sklearn.preprocessing import KBinsDiscretizer, PolynomialFeatures

4

## Procesamiento de Datos por Fragmentos

Para optimizar el uso de memoria RAM frente a un volumen masivo de datos, el algoritmo lee el archivo segmentando la información en lotes (`chunks`). Durante esta fase se calculan las estadísticas globales necesarias para la posterior discretización y el escalado de la función objetivo.

In [ ]:
# --- Paso 1: Carga y Procesamiento de Datos ---
print("\nCargando y procesando TODOS los datos de entrenamiento...")

# Se define la ruta del archivo con los datos del muestreo
ruta_csv = "muestreo_de_puntos_SOBOL.csv"

nuevos_nombres = [f'x{i}' for i in range(1, 11)] + ['fun']

print("Calculando estadísticas globales...")
x_mins = np.full(10, np.inf)
x_maxs = np.full(10, -np.inf)
y_min_global = np.inf
y_max_global = -np.inf
sample_y = []
total_filas = 0

semilla_aleatoria = int(np.random.randint(1, 100000))
with open("semilla.txt", "w") as f:
    f.write(str(semilla_aleatoria))

for chunk in pd.read_csv(ruta_csv, chunksize = 50000):
    chunk = chunk.sample(frac=0.8, random_state=semilla_aleatoria)
    if len(chunk.columns) == 11:
        chunk.columns = nuevos_nombres
    chunk = chunk.dropna()
    total_filas += len(chunk)

    y_vals = chunk['fun'].values
    y_min_global = min(y_min_global, y_vals.min())
    y_max_global = max(y_max_global, y_vals.max())

    sample_y.append(y_vals[::100])

    X_chunk = chunk[nuevos_nombres[:10]].values
    x_mins = np.minimum(x_mins, X_chunk.min(axis=0))
    x_maxs = np.maximum(x_maxs, X_chunk.max(axis=0))

percentil_5 = np.percentile(np.concatenate(sample_y), 20)
y_range_original = y_max_global - y_min_global
y_min_original = y_min_global
if y_range_original == 0: y_range_original = 1
print(f"Procesamiento de datos finalizado. Se usarán {total_filas} puntos para el ajuste.")
del sample_y

## Ajuste del Modelo Matemático

El núcleo algorítmico configura un problema de optimización convexa utilizando la biblioteca `cvxpy`. Se generan variables binarias que representan los intervalos del espacio continuo y se construyen las interacciones cuadráticas necesarias para la matriz QUBO.

In [4]:
# --- Paso 2: Función de Ajuste PONDERADO ---
def ajustar_modelo_estable():
    print("\nIniciando ajuste de modelo PONDERADO para precisión, modificar segun se requiera...")
    pesos_valor_alto = 1.0
    pesos_valor_bajo = 1.0

    print(f"Aplicando pesos: puntos con fun <= {percentil_5:.4f} tendrán peso: {pesos_valor_alto}.")

    primeras_5_vars = [f'x{i}' for i in range(1, 6)]
    ultimas_5_vars = [f'x{i}' for i in range(6, 11)]

    print("Iniciando discretización de variables...")
    discretizer_primeras = KBinsDiscretizer(n_bins=17, encode='onehot-dense', strategy='uniform')
    discretizer_ultimas = KBinsDiscretizer(n_bins=15, encode='onehot-dense', strategy='uniform')
    discretizer_primeras.fit(np.vstack([x_mins[:5], x_maxs[:5]]))
    discretizer_ultimas.fit(np.vstack([x_mins[5:], x_maxs[5:]]))

    print("Discretización completada. Generando características polinomiales...")
    num_binarias = 5*17 + 5*15
    nombres_binarias_sk = [f'x{i}' for i in range(num_binarias)]
    poly = PolynomialFeatures(degree=2, include_bias=True)
    poly.fit(np.zeros((1, num_binarias)))

    print("Características generadas. Configurando problema de optimización CVXPY...")
    d = poly.n_output_features_
    A = np.zeros((d, d))
    b_vec = np.zeros(d)
    lambda_reg = 1.0

    print("Acumulando XᵀWX y XᵀWy por lotes...")
    for chunk in pd.read_csv(ruta_csv, chunksize = 50000):
        chunk = chunk.sample(frac=0.8, random_state=semilla_aleatoria)
        if len(chunk.columns) == 11:
            chunk.columns = nuevos_nombres
        chunk = chunk.dropna()

        y_actual = chunk['fun'].values
        y_scaled = (y_actual - y_min_original) / y_range_original
        pesos = np.where(y_actual <= percentil_5, pesos_valor_alto, pesos_valor_bajo)

        X_chunk = chunk[primeras_5_vars + ultimas_5_vars]
        Xb_primeras = discretizer_primeras.transform(X_chunk[primeras_5_vars])
        Xb_ultimas = discretizer_ultimas.transform(X_chunk[ultimas_5_vars])
        X_binarias = np.hstack([Xb_primeras, Xb_ultimas])
        X_poly = poly.transform(X_binarias)

        X_poly_w = X_poly * pesos[:, None]
        A += X_poly.T @ X_poly_w
        b_vec += X_poly.T @ (pesos * y_scaled)
        del X_poly_w
        del X_poly, Xb_primeras, Xb_ultimas, X_binarias, X_chunk
        gc.collect()

    A = (A + A.T) / 2.0
    coefs = cp.Variable(d)
    objective = cp.Minimize(cp.quad_form(coefs, cp.psd_wrap(A)) - 2 * b_vec @ coefs + lambda_reg * cp.sum_squares(coefs))
    problem = cp.Problem(objective)
    print("Iniciando resolución del solver (esto puede tardar unos minutos)...")
    problem.solve(solver=cp.OSQP, verbose=False)
    print("Resolución del solver completada.")

    if problem.status not in ["optimal", "optimal_inaccurate"]:
        raise ValueError("El problema de optimización del ajuste falló en su resolución.")
    print("Ajuste ponderado estable completado.")

    final_coefficients = coefs.value
    return {
        'intercept': final_coefficients[0],
        'coefs': final_coefficients[1:],
        'poly_feature_names': poly.get_feature_names_out(nombres_binarias_sk),
        'y_min': y_min_original,
        'y_range': y_range_original,
        'n_bins_primeras': 17,
        'n_bins_ultimas': 15,
        'num_vars_binarias': num_binarias,
        'discretizer_primeras': discretizer_primeras,
        'discretizer_ultimas': discretizer_ultimas
    }

# --- Paso 3: Ejecución del Ajuste y Guardado del PKL ---
modelo_info = ajustar_modelo_estable()

ruta_guardado_modelo = "modelo_10D_entrenado_.pkl"
print(f"\nGuardando el modelo ajustado completo en: {ruta_guardado_modelo}")
try:
    with open(ruta_guardado_modelo, 'wb') as f:
        pickle.dump(modelo_info, f)
    print("Modelo guardado exitosamente en Disco.")
except Exception as e:
    print(f"Error al guardar el modelo: {e}")

del modelo_info

Modelo guardado exitosamente en Disco.
